# Chapter 18: Fine-Tuning Locally (Hugging Face + Unsloth)
## Topic 5: QLoRA Specifically

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 4 established LoRA's mechanism precisely: freeze the base weight matrix W, train only a small, low-rank update (B×A). But Topic 3's own real memory math showed the *frozen base model itself* still requires 4GB even after LoRA's dramatic parameter-count reduction — that 4GB figure came specifically from loading the model in **4-bit quantization**, a detail Topic 4 didn't yet explain. This topic opens that specific box: what quantization actually does mathematically, and how combining it with LoRA (the resulting combination is precisely what "QLoRA" names) is what makes Topic 3's full, real 4.35GB total genuinely achievable.
- **Quantization**, precisely: representing a number using fewer bits than its original precision. A model's weights are typically stored as 16-bit floating-point numbers (Topic 1's own fp16 memory calculations); 4-bit quantization instead represents each weight using only 4 bits — a 4x reduction in the memory needed to store the frozen base model's weights, directly explaining Topic 3's real, computed 4GB base-model figure (8 billion parameters × 0.5 bytes each, exactly the 4-bit-per-parameter math).
- **QLoRA** is precisely the combination this project's actual configuration uses: the frozen base model W is stored in this heavily quantized, 4-bit form (dramatically reducing its own memory footprint), while LoRA's trainable low-rank update matrices (B and A, Topic 4's mechanism) are still trained and stored in higher precision (typically 16-bit) — since these are tiny compared to the full model, keeping them in higher precision costs very little extra memory while preserving the fine-tuning process's ability to learn an accurate, high-quality update.


### 2. Internal Working — Step by Step

**The actual mechanics of QLoRA, step by step:**

1. **Load the frozen base model's weights in 4-bit quantized form** — each weight, originally a 16-bit or 32-bit floating-point number, gets mapped to one of only 16 possible discrete values (since 4 bits can represent 2⁴ = 16 distinct values) — a real, computable reduction in precision, but one specifically designed (via a technique called NF4, or 4-bit NormalFloat, in QLoRA's actual published method) to minimize the accuracy lost in this process for typical neural network weight distributions specifically.
2. **During the forward pass (both training and inference), the quantized weights are "dequantized" back to a higher-precision format on the fly**, specifically for the actual matrix multiplication computation — this on-the-fly dequantization is what allows the memory savings of 4-bit storage while still performing the numerically-sensitive computation itself in higher precision, avoiding compounding quantization errors during the actual forward pass computation.
3. **LoRA's trainable matrices (B and A, Topic 4's mechanism) are added on top of this quantized, frozen base**, and are themselves trained and stored in standard higher precision (typically bf16) — since these matrices are, per Topic 4's own real math, only about 0.37% of the full model's parameter count, keeping them in higher precision costs very little extra memory while preserving training quality for the update itself.
4. **Gradients only ever need to be computed and stored for the LoRA matrices, never for the frozen, quantized base weights** — this is exactly Topic 4's own core efficiency mechanism, now combined with quantization's separate, additional memory savings specifically on the frozen base model's own storage footprint, together producing Topic 3's real, complete 4.35GB total.


### 3. How This Is Implemented in This Project

- Topic 3's real, actual Unsloth code specifies `load_in_4bit = True` at the model-loading step — this is precisely QLoRA's quantization mechanism being applied, loading Llama-3.1-8B-Instruct's frozen base weights in exactly the 4-bit form this topic describes, directly producing Topic 3's real, computed 4GB base-model memory figure (8 billion parameters × 0.5 bytes/parameter).
- Topic 3's model checkpoint name itself — `"unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"` — is a pre-quantized checkpoint, meaning the 4-bit quantization has already been applied to these weights and made available for direct loading, rather than requiring this project to perform the quantization computation itself from a full-precision checkpoint at load time.
- This project's complete, real QLoRA configuration — Topic 3's 4-bit-quantized base model combined with Topic 4's rank-16 LoRA adapter — is precisely what Topic 3's own real, computed 4.35GB total training memory figure represents: quantization's savings on the frozen base (Topic 5's specific contribution) combined with LoRA's savings on trainable-parameter count (Topic 4's specific contribution), together producing the full, real 22x reduction versus full fine-tuning that makes this project's fine-tuning goal achievable on its actual RTX 4060 hardware.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Quantization is not free — it introduces a real, measurable precision loss, even with NF4's specific design to minimize this for typical weight distributions** — this needs empirical validation (does the quantized base model still perform reasonably on straightforward tasks before any fine-tuning is even applied) rather than an assumption that 4-bit quantization has zero meaningful effect on model quality.
- **The on-the-fly dequantization step during the forward pass adds real, if generally small, computational overhead compared to a model already stored in the higher precision needed for computation** — this is a genuine speed-vs-memory trade-off: QLoRA saves substantial memory at the cost of some additional computation during every forward pass, compared to a hypothetical (but memory-infeasible, per Topic 1) higher-precision-throughout approach.
- **Not every quantization scheme is equally well-suited to neural network weights specifically** — QLoRA's specific NF4 format was deliberately designed and empirically validated for this purpose (typical neural network weight distributions), which is why this project uses QLoRA's specific, validated approach rather than a more generic, naive quantization scheme that might introduce more precision loss for this specific kind of data.
- **Debugging unexpectedly poor model behavior after QLoRA fine-tuning requires distinguishing a quantization-precision problem from a LoRA-rank problem (Topic 4's concern) or a training-hyperparameter problem (Topic 6)** — a quantization-specific issue would show up even before any LoRA fine-tuning is applied at all (checking the quantized base model's own baseline behavior, prior to training, isolates this specific concern from the other two categories).
- **Monitoring:** comparing the quantized base model's behavior on a small set of straightforward test cases against its original, full-precision behavior (before fine-tuning is even applied) is the direct, practical way to confirm quantization itself isn't introducing an unacceptable quality regression, separate from whatever the subsequent LoRA fine-tuning process might contribute.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **4-bit quantization (this project's actual, real configuration) vs 8-bit quantization:** 4-bit provides greater memory savings (Topic 3's real, computed 4GB figure would be roughly double, 8GB, at 8-bit precision) at a somewhat greater precision-loss risk; 8-bit is a more conservative middle ground — the right choice depends on how tightly memory-constrained the specific hardware target actually is (this project's real 8GB VRAM strongly favors 4-bit, given Topic 3's own demonstrated tight margins) and how much precision loss is empirically observed to be acceptable for the specific task.
- **QLoRA (quantized base + higher-precision LoRA adapter) vs a hypothetical fully-quantized approach (quantizing the LoRA adapter too):** keeping the LoRA adapter in higher precision (this project's actual approach) costs very little extra memory, given the adapter's tiny relative parameter count (Topic 4's real 0.37% figure), while preserving training quality for the update itself — fully quantizing the tiny adapter too would save negligible additional memory while risking meaningfully degrading the fine-tuning process's ability to learn an accurate update.
- **Using a pre-quantized checkpoint (this project's actual approach) vs quantizing a full-precision checkpoint at load time:** using a pre-quantized checkpoint (Topic 3's actual model name) is simpler and faster to get started with; quantizing at load time from a full-precision source gives more control over the specific quantization process but requires additional computation and disk space for the original, full-precision weights before quantization — the pre-quantized approach is the more practical default when a well-validated, pre-quantized checkpoint is already available, as it is for this project's actual chosen model.


### 6. Alternatives and When to Use Each

- **Full-precision (fp16/bf16) base model, LoRA adapter only (no quantization):** would still provide LoRA's parameter-efficiency benefit, but without quantization's additional memory savings on the base model itself — insufficient for this project's genuinely tight 8GB constraint, given Topic 1's own math showing even a 16GB base-model footprint alone would consume the entire available budget.
- **QLoRA — quantized base model, higher-precision LoRA adapter (this project's actual, real configuration):** the right choice for this project's genuinely constrained hardware, combining quantization's base-model memory savings with LoRA's parameter-efficiency savings for the combined, real 22x total reduction Topic 3 demonstrated.
- **8-bit quantization instead of 4-bit:** a more conservative alternative worth considering if 4-bit's precision loss is empirically found to be unacceptable for a specific task, at the cost of roughly double the base-model memory footprint.


### 7. Common Mistakes and Production Failures

- Assuming quantization has zero meaningful effect on model quality without empirically validating the quantized base model's own baseline behavior before fine-tuning.
- Conflating a quantization-precision problem with a LoRA-rank problem or a training-hyperparameter problem when diagnosing unsatisfactory fine-tuned model behavior, rather than isolating each category through targeted, separate checks.
- Not accounting for the real, if generally modest, computational overhead quantization's on-the-fly dequantization step adds during every forward pass, when reasoning about training or inference speed.
- Quantizing the LoRA adapter matrices themselves (rather than keeping them in higher precision), sacrificing training quality for a negligible additional memory saving given the adapter's already-tiny parameter count.
- Choosing a generic, naive quantization scheme rather than a specifically validated approach like QLoRA's NF4 format, risking greater precision loss than necessary for typical neural network weight distributions.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What does QLoRA add on top of Topic 4's LoRA mechanism?
  A: QLoRA specifically stores the frozen base model's weights in 4-bit quantized form, dramatically reducing the base model's own memory footprint, while keeping LoRA's small, trainable low-rank update matrices in higher precision. This combines quantization's savings on the (much larger) frozen base model with LoRA's savings on trainable-parameter count, together producing far greater total memory reduction than either technique alone.

- Q: Why does Topic 3's real base-model memory figure come out to exactly 4GB for an 8-billion-parameter model?
  A: At 4-bit quantization, each parameter requires 0.5 bytes of storage (4 bits = 0.5 bytes) — 8 billion parameters × 0.5 bytes/parameter = 4 billion bytes, or 4GB, exactly matching Topic 3's real, computed figure and directly demonstrating quantization's precise, calculable memory-reduction effect.

**Intermediate**

- Q: Why are LoRA's adapter matrices kept in higher precision rather than also being quantized to 4 bits?
  A: The adapter matrices are, per Topic 4's own real math, only about 0.37% of the full model's parameter count — quantizing them further would save a negligible amount of additional memory while risking meaningfully degrading the fine-tuning process's ability to learn an accurate, high-quality update. The memory cost of keeping this tiny fraction of parameters in higher precision is small, making it a clearly favorable trade-off.

- Q: What is "on-the-fly dequantization," and why is it necessary?
  A: During the forward pass, quantized 4-bit weights are temporarily converted back to a higher-precision format specifically for the actual matrix multiplication computation, rather than performing the computation directly in 4-bit precision — this avoids compounding numerical errors during the sensitive computation itself, while still benefiting from 4-bit storage's memory savings when the weights aren't actively being computed with.

**Advanced**

- Q: Design an empirical validation process to confirm 4-bit quantization isn't introducing an unacceptable quality regression for this project's specific base model, independent of the subsequent LoRA fine-tuning process.
  A: Before applying any LoRA fine-tuning, run the quantized base model (Topic 3's actual `load_in_4bit=True` configuration) against a small set of straightforward, already-understood test cases (perhaps a subset of Chapter 1's original classification examples), and compare its behavior against the model's known, expected baseline behavior — ideally against the same model in its original, non-quantized precision if available for comparison. If the quantized model's baseline behavior is comparable, quantization's precision loss is confirmed empirically acceptable for this specific model and task category; if meaningfully degraded, this points toward reconsidering 4-bit quantization (perhaps 8-bit instead) before proceeding to the more expensive LoRA fine-tuning process on top of it.

- Q: A teammate argues that since quantization introduces some precision loss, this project should avoid it and use a higher-precision base model despite the memory cost, finding a different way to fit within 8GB VRAM. What would you weigh in response?
  A: Given Topic 1's own real math (a full-precision 8B model alone requires 16GB just for weights, already double this project's entire 8GB budget), avoiding quantization entirely isn't a viable alternative for this project's specific hardware constraint — some form of memory reduction beyond LoRA's parameter efficiency alone is necessary, and QLoRA's specific, empirically-validated 4-bit approach (NF4, designed specifically to minimize precision loss for neural network weights) is a well-established, reasonable choice for achieving this, rather than an arbitrary or excessively risky corner-cutting measure.

**Scenario-based**

- Q: After loading this project's actual QLoRA configuration and running a quick baseline test (no fine-tuning yet), the quantized model's responses seem noticeably less coherent than expected for an 8B-parameter instruction-tuned model. Walk through your diagnostic process.
  A: This is exactly the kind of pre-fine-tuning baseline check this topic's theory recommends — since no LoRA training has occurred yet, this rules out both Topic 4's rank-related concerns and Topic 6's hyperparameter concerns entirely, isolating the issue specifically to the quantization step or a possible loading/configuration error. Check whether the specific pre-quantized checkpoint being loaded is genuinely correct and complete (a corrupted or mismatched checkpoint could produce exactly this symptom), and if the checkpoint is confirmed correct, this would be a signal to reconsider whether 4-bit quantization is producing an unacceptable precision loss for this specific model, potentially warranting a switch to 8-bit quantization instead.

**Follow-up questions to expect:**

- "How would you decide between 4-bit and 8-bit quantization if you had slightly more VRAM headroom available than this project's exact 8GB?"
- "What would change about this approach if this project needed to fine-tune a model where a well-validated, pre-quantized checkpoint wasn't already available?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Quantization is a general numerical-representation technique used broadly across computing, not invented specifically for neural network fine-tuning** — the same core idea (representing values with fewer bits, trading precision for memory/storage efficiency) appears in image and audio compression, embedded systems programming, and numerical computing generally; QLoRA applies this general principle specifically and carefully to neural network weight distributions.
- **NF4 (4-bit NormalFloat)'s specific design — accounting for the actual statistical distribution of neural network weights, rather than using a naive, uniform quantization scheme — is itself a meaningful, published technical contribution**, not an incidental implementation detail; recognizing that quantization schemes can be more or less well-suited to a specific kind of data (here, weights that tend to follow an approximately normal distribution) is a valuable, generalizable insight.
- **This topic's combination of Topic 4's LoRA mechanism with quantization is the complete, real technique underlying Topic 3's actual configuration** — understanding both topics together is what makes Topic 6's upcoming hyperparameter discussion (rank, alpha, learning rate, and quantization-related settings together) fully concrete, rather than a list of configuration values without clear grounding in what each one actually does.

### 10. Quick Revision Summary (for last-minute interview prep)

> QLoRA combines Topic 4's LoRA mechanism (freezing the base weight matrix, training only a small, low-rank update) with quantization of the frozen base model itself — storing its weights in 4-bit precision rather than the standard 16-bit, directly explaining Topic 3's real, computed 4GB base-model memory figure for an 8-billion-parameter model (8 billion × 0.5 bytes/parameter at 4-bit precision). During the forward pass, quantized weights are temporarily dequantized to higher precision specifically for the actual computation, avoiding compounding numerical errors while still benefiting from 4-bit storage's memory savings. LoRA's own small adapter matrices remain in higher precision throughout, since their tiny relative parameter count (Topic 4's real 0.37% figure) makes this a clearly favorable trade-off, preserving training quality for the actual learned update. QLoRA's specific quantization approach (NF4) was deliberately designed and empirically validated for typical neural network weight distributions, minimizing precision loss compared to a naive, generic quantization scheme — though quantization is never entirely free, and its effect on model quality should be empirically validated (checking the quantized base model's own behavior before any fine-tuning is applied) rather than assumed away. Combined, quantization's base-model savings and LoRA's parameter-efficiency savings together produce Topic 3's real, complete 4.35GB total training memory figure — the concrete, computed reason this project's fine-tuning goal is achievable at all on its real, constrained RTX 4060 hardware.


### Module 1: Real Quantization Math — Simulating 4-bit Weight Storage

In [1]:

# ------------------------------------------------------------------
# MODULE 1: REAL quantization math, using actual NumPy computation
# ------------------------------------------------------------------

import numpy as np

np.random.seed(42)

# REAL simulated weight values -- neural network weights typically
# follow an approximately normal distribution, exactly what NF4 is
# specifically designed to represent well
simulated_weights = np.random.randn(10000) * 0.02  # typical small-magnitude weight values

def simulate_uniform_quantization(values, num_bits):
    # REAL, computable quantization -- maps continuous values to a
    # fixed number of discrete levels (2^num_bits), simulating the
    # core mechanism (a simplified UNIFORM quantization, standing in for
    # QLoRA's more sophisticated NF4 scheme, which is specifically
    # optimized for normally-distributed weights).
    num_levels = 2 ** num_bits
    min_val, max_val = values.min(), values.max()
    step_size = (max_val - min_val) / (num_levels - 1)
    quantized_indices = np.round((values - min_val) / step_size)
    dequantized = quantized_indices * step_size + min_val
    return dequantized

def quantization_error(original, quantized):
    return np.linalg.norm(original - quantized) / np.linalg.norm(original)

print("=" * 70)
print("MODULE 1: REAL QUANTIZATION MATH")
print("=" * 70)
print(f"\nSimulated weight values: {len(simulated_weights):,} values")
print(f"Original value range: [{simulated_weights.min():.4f}, {simulated_weights.max():.4f}]")

for num_bits in [2, 4, 8, 16]:
    quantized = simulate_uniform_quantization(simulated_weights, num_bits)
    error = quantization_error(simulated_weights, quantized)
    num_levels = 2 ** num_bits
    print(f"\n{num_bits}-bit quantization ({num_levels} discrete levels):")
    print(f"  Relative reconstruction error: {error:.4f}")

print("\nCONFIRMED: quantization error DECREASES as bit-depth increases (more")
print("discrete levels available to represent the original continuous")
print("values) -- this is the REAL, computable precision-vs-memory")
print("trade-off QLoRA's 4-bit choice represents.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL QUANTIZATION MATH

Simulated weight values: 10,000 values
Original value range: [-0.0784, 0.0785]

2-bit quantization (4 discrete levels):
  Relative reconstruction error: 0.7783

4-bit quantization (16 discrete levels):
  Relative reconstruction error: 0.1515

8-bit quantization (256 discrete levels):
  Relative reconstruction error: 0.0089

16-bit quantization (65536 discrete levels):
  Relative reconstruction error: 0.0000

CONFIRMED: quantization error DECREASES as bit-depth increases (more
discrete levels available to represent the original continuous
values) -- this is the REAL, computable precision-vs-memory
trade-off QLoRA's 4-bit choice represents.

Module 1 complete. Run Module 2 next.


### Module 2: Real Memory Savings — 4-bit vs 8-bit vs 16-bit, Verified Against Topic 3

In [2]:

# ------------------------------------------------------------------
# MODULE 2: REAL memory savings math, verified against Topic 3's figure
# ------------------------------------------------------------------

MODEL_PARAMS_BILLIONS = 8.0

def compute_base_model_memory_gb(params_billions, bits_per_param):
    bytes_per_param = bits_per_param / 8
    return (params_billions * 1e9 * bytes_per_param) / 1e9

print("=" * 70)
print("MODULE 2: REAL MEMORY FOOTPRINT ACROSS QUANTIZATION LEVELS")
print("=" * 70)
print(f"\n{'Precision':<15} | {'Bits/Param':>10} | {'Base Model Memory':>18}")
print("-" * 50)

for label, bits in [("fp16 (Topic 1)", 16), ("8-bit", 8), ("4-bit (Topic 3)", 4)]:
    memory_gb = compute_base_model_memory_gb(MODEL_PARAMS_BILLIONS, bits)
    print(f"{label:<15} | {bits:>10} | {memory_gb:>17.2f} GB")

memory_4bit = compute_base_model_memory_gb(MODEL_PARAMS_BILLIONS, 4)
matches_topic3 = abs(memory_4bit - 4.0) < 0.01
print(f"\n4-bit computed memory ({memory_4bit:.2f} GB) matches Topic 3's real figure (4.00 GB)? {matches_topic3}")

memory_fp16 = compute_base_model_memory_gb(MODEL_PARAMS_BILLIONS, 16)
reduction_vs_fp16 = memory_fp16 / memory_4bit
print(f"\n4-bit quantization provides a {reduction_vs_fp16:.0f}x memory reduction vs fp16,")
print(f"for the BASE MODEL WEIGHTS ALONE -- this is Topic 5's SPECIFIC")
print(f"contribution, separate from and additional to Topic 4's LoRA")
print(f"parameter-efficiency savings on the TRAINABLE parameters.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: REAL MEMORY FOOTPRINT ACROSS QUANTIZATION LEVELS

Precision       | Bits/Param |  Base Model Memory
--------------------------------------------------
fp16 (Topic 1)  |         16 |             16.00 GB
8-bit           |          8 |              8.00 GB
4-bit (Topic 3) |          4 |              4.00 GB

4-bit computed memory (4.00 GB) matches Topic 3's real figure (4.00 GB)? True

4-bit quantization provides a 4x memory reduction vs fp16,
for the BASE MODEL WEIGHTS ALONE -- this is Topic 5's SPECIFIC
contribution, separate from and additional to Topic 4's LoRA
parameter-efficiency savings on the TRAINABLE parameters.

Module 2 complete. Run Module 3 next.


### Module 3: The Complete QLoRA Picture — Quantization + LoRA Together

In [3]:

# ------------------------------------------------------------------
# MODULE 3: The COMPLETE QLoRA picture, combining Topics 4 and 5
# ------------------------------------------------------------------

# Topic 4's real LoRA parameter count
REAL_HIDDEN_DIM = 4096
REAL_RANK = 16
REAL_NUM_TARGET_MODULES = 7
REAL_NUM_LAYERS = 32
lora_params_per_module = 2 * REAL_RANK * REAL_HIDDEN_DIM
total_lora_params = lora_params_per_module * REAL_NUM_TARGET_MODULES * REAL_NUM_LAYERS

# Topic 5's real quantization-based base model memory
base_model_4bit_gb = compute_base_model_memory_gb(MODEL_PARAMS_BILLIONS, 4)

# LoRA adapter memory (Topic 4's tiny parameter count, kept in fp16)
lora_weights_gb = (total_lora_params * 2) / 1e9
lora_gradients_gb = (total_lora_params * 2) / 1e9
lora_optimizer_gb = (total_lora_params * 8) / 1e9

total_qlora_memory = base_model_4bit_gb + lora_weights_gb + lora_gradients_gb + lora_optimizer_gb

# comparison: full fine-tuning (Topic 1's real math)
full_finetuning_memory = compute_base_model_memory_gb(MODEL_PARAMS_BILLIONS, 16) * 6  # weights+grads+2x optimizer states (fp16=2, but optimizer fp32=4+4)

print("=" * 70)
print("MODULE 3: THE COMPLETE QLoRA PICTURE (Topic 4 + Topic 5 combined)")
print("=" * 70)
print(f"\nTopic 5's contribution -- quantized base model: {base_model_4bit_gb:.2f} GB")
print(f"Topic 4's contribution -- LoRA weights:          {lora_weights_gb:.3f} GB")
print(f"                       -- LoRA gradients:         {lora_gradients_gb:.3f} GB")
print(f"                       -- LoRA optimizer states:  {lora_optimizer_gb:.3f} GB")
print(f"\nTOTAL QLoRA memory (Topic 4 + Topic 5 together): {total_qlora_memory:.2f} GB")

matches_topic3_total = abs(total_qlora_memory - 4.35) < 0.01
print(f"Matches Topic 3's real, complete total (4.35 GB)? {matches_topic3_total}")

print(f"\nFull fine-tuning memory (Topic 1's approach, for comparison): {full_finetuning_memory:.0f} GB")
reduction = full_finetuning_memory / total_qlora_memory
print(f"QLoRA's TOTAL reduction vs full fine-tuning: {reduction:.0f}x")

print("\nModule 3 complete. All Chapter 18 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 18 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("Real quantization math demonstrated concretely -- error decreases")
print("as bit-depth increases, the actual, computable precision/memory")
print("trade-off underlying QLoRA's specific 4-bit choice.")
print()
print("Real memory math VERIFIED against Topic 3's own figure: 4-bit")
print("quantization produces EXACTLY 4.00 GB for this project's real 8B")
print("model, a 4x reduction vs fp16's 16 GB, for the base model alone.")
print()
print("The COMPLETE QLoRA picture (Topic 4's LoRA + Topic 5's")
print("quantization together) reproduces Topic 3's exact real total")
print("(4.35 GB) -- INDEPENDENTLY VERIFIED by combining both topics'")
print("separate mathematical contributions.")
print()
print("This complete, verified mathematical foundation is the direct")
print("prerequisite for Topic 6's hyperparameter discussion, which")
print("covers rank, alpha, learning rate, and related settings that")
print("configure this exact mechanism in practice.")


MODULE 3: THE COMPLETE QLoRA PICTURE (Topic 4 + Topic 5 combined)

Topic 5's contribution -- quantized base model: 4.00 GB
Topic 4's contribution -- LoRA weights:          0.059 GB
                       -- LoRA gradients:         0.059 GB
                       -- LoRA optimizer states:  0.235 GB

TOTAL QLoRA memory (Topic 4 + Topic 5 together): 4.35 GB
Matches Topic 3's real, complete total (4.35 GB)? True

Full fine-tuning memory (Topic 1's approach, for comparison): 96 GB
QLoRA's TOTAL reduction vs full fine-tuning: 22x

Module 3 complete. All Chapter 18 Topic 5 modules done.

CHAPTER 18 TOPIC 5 -- KEY POINTS TO REMEMBER
Real quantization math demonstrated concretely -- error decreases
as bit-depth increases, the actual, computable precision/memory
trade-off underlying QLoRA's specific 4-bit choice.

Real memory math VERIFIED against Topic 3's own figure: 4-bit
quantization produces EXACTLY 4.00 GB for this project's real 8B
model, a 4x reduction vs fp16's 16 GB, for the base model